# 04 - Cross-encoder reranking

First-stage retriever (here: hybrid BM25+dense) pulls a wide candidate pool,
then a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) jointly scores
each `(query, chunk)` pair with full attention. Highest precision, at the
cost of one extra forward pass per candidate.

In [1]:
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv

from rag.data_ingestion import chunk_documents, load_documents
from rag.llm import LLM
from rag.metrics import embedding_faithfulness
from rag.pipeline import RAGPipeline
from rag.retrievers import HybridRetriever, RerankerRetriever

warnings.filterwarnings('ignore')
load_dotenv(os.path.abspath('../.env'))

FILE_PATH = '../data/google_10K.pdf'
QUERY = 'Summarize the principal risk factors that the company discloses related to artificial intelligence, competition, and regulation, and group them by category.'

docs = load_documents(FILE_PATH)
chunks = chunk_documents(docs, chunk_size=2000, chunk_overlap=200)
print(f'Loaded {len(docs)} pages -> {len(chunks)} chunks')

Loaded 107 pages -> 230 chunks


In [2]:
retriever = RerankerRetriever(
    base_retriever=HybridRetriever(),
    candidate_pool=20,
)
retriever.add_documents(chunks)

for i, hit in enumerate(retriever.retrieve(QUERY, k=3), start=1):
    snippet = hit.document.page_content[:200].replace('\n', ' ')
    print(f'{i}. (CE={hit.score:.3f}) {snippet}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. (CE=-1.350) cause or contribute to such differences include, but are not limited to, those discussed in this Annual Report on Form 10-K, including the risks discussed in Part I, Item 1A "Risk Factors" and the tre...
2. (CE=-3.407) Competition for Specified Smartphone Software in Japan; regulations and legal settlements in the US, South Korea, and elsewhere that affect Google Play's billing policies, fees, and business model; as...
3. (CE=-4.479) Table of Contents Alphabet Inc. ITEM 1A. RISK FACTORS Our operations and financial results are subject to various risks and uncertainties, including but not limited to those described below, which cou...


In [3]:
llm = LLM(api_key=os.environ['GROQ_API_KEY']).get_llm(
    provider='groq',
    model_name='llama-3.3-70b-versatile',
    temperature=0.0,
)
pipeline = RAGPipeline(retriever=retriever, llm=llm, top_k=8)
response = pipeline.answer(QUERY)
print(response.answer)

Based on the provided context, the principal risk factors disclosed by the company related to artificial intelligence, competition, and regulation can be summarized and grouped by category as follows:

**Artificial Intelligence (AI) Risks:**
- Risks related to the development, use, and provision of AI technologies and other digital products and services, which could result in monetary penalties or other regulatory actions (Doc 2, page 20).
- AI-enabled products and services may give rise to risks related to harmful content, inaccuracies, discrimination, intellectual property infringement or misappropriation, violation of rights of publicity, defamation, data privacy, cybersecurity, minor protection, and other issues (Doc 6, page 15).
- Unintended consequences, uses, or customization of AI tools and systems may negatively affect human rights, privacy, employment, or other social concerns (Doc 6, page 15).

**Competition Risks:**
- Competition for specified smartphone software in Japan (

## Inline evaluation

`embedding_faithfulness` confirms the LLM stuck to the chunks the
cross-encoder surfaced.

In [4]:
context_strings = [doc.page_content for doc in response.contexts]
faith = embedding_faithfulness(response.answer, context_strings)
print(f'embedding_faithfulness = {faith:.3f}')

embedding_faithfulness = 0.787
